In [1]:
import gzip
import json
import pandas as pd

# TASK 1

In [ ]:
def load_amazon_data(file_path):
    data = []
    
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for line in f:
            review = json.loads(line)
            
            extracted = {
                'user_id': review.get('reviewerID'),
                'asin': review.get('asin'),
                'rating': review.get('overall'),
                'timestamp': review.get('unixReviewTime')
            }
            data.append(extracted)
            
    df = pd.DataFrame(data)
    
    return df

In [ ]:
df_full = load_amazon_data('Sports_and_Outdoors.json.gz')
print(df_full.head())
print(df_full.info()) # To pokaże ile RAMu zużywa

In [ ]:
def get_5_core(df):
    print(f"Rozmiar początkowy: {len(df)}")
    
    df_core = df.copy() 
    
    while True:
        start_len = len(df_core)
        
        # Filtrujemy przedmioty - zostawaimy te co mają >= 5 ocen
        item_counts = df_core['asin'].value_counts()
        valid_items = item_counts[item_counts >= 5].index
        df_core = df_core[df_core['asin'].isin(valid_items)]
        
        # Filtrujemy użytkowników - zostawiamy tylko co ocenili >= 5 przedmiotów (z tych przedmiotów co zostały)
        user_counts = df_core['user_id'].value_counts()
        valid_users = user_counts[user_counts >= 5].index
        df_core = df_core[df_core['user_id'].isin(valid_users)]
        
        # Robimy to w pętli, bo po wyrzuceniu użytnowników, niektóre przedmioty mogły też odpaść
        # Jeśli jest stabilnie po jednym ruchu to możemy kończyć
        end_len = len(df_core)
        if start_len == end_len:
            break 
            
    print(f"Rozmiar 5-core: {len(df_core)}")
    return df_core

In [ ]:
df_5core = get_5_core(df_full)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def analyze_dataset(df, dataset_name):
    print(f"Analiza: {dataset_name}")
    
    num_interactions = len(df)
    num_users = df['user_id'].nunique()
    num_items = df['asin'].nunique()
    
    print(f"Liczba interakcji: {num_interactions:,}")
    print(f"Unikalni użytkownicy: {num_users:,}")
    print(f"Unikalne przedmioty: {num_items:,}")
    
    # (c) Sparsity - jakiej cześci wszystkich możliwych interkacji nam brakuje
    possible_interactions = num_users * num_items
    density = num_interactions / possible_interactions
    sparsity = 1.0 - density
    print(f"Sparsity: {sparsity:.6f} (Gęstość macierzy: {density * 100:.4f}%)")
    
    # (d) Duplikaty i sprzeczne dane
    # Szukamy sytuacji, gdzie ten sam użytkownik ocenił ten sam przedmiot więcej niż raz
    duplicates_mask = df.duplicated(subset=['user_id', 'asin'], keep=False)
    num_duplicates = duplicates_mask.sum()
    print(f"Liczba zduplikowanych ocen użytkownika: {num_duplicates:,}")
    
    if num_duplicates > 0:
        # sprzeczne, tzn różne oceny dla tej samej pary
        dup_df = df[duplicates_mask]
        contradictory = dup_df.groupby(['user_id', 'asin'])['rating'].nunique()
        num_contradictory = (contradictory > 1).sum()
        print(f"A w tym sprecznych: {num_contradictory:,}")
    
    print("-" * 50)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"EDA: {dataset_name}", fontsize=16)
    
    # (a) Liczba interkacji per przedmiot
    item_counts = df['asin'].value_counts().values
    axes[0].plot(item_counts, color='blue')
    axes[0].set_title("(a) Interakcje per przedmiot")
    axes[0].set_xlabel("Przedmioty (posortowane wg popularności)")
    axes[0].set_ylabel("Liczba ocen (log)")
    axes[0].set_yscale('log')
    axes[0].grid(True, linestyle=':', alpha=0.6)
    
    # (b) Rozkład interakcji użytkowników
    user_counts = df['user_id'].value_counts().values
    sns.histplot(user_counts, bins=50, ax=axes[1], color='orange', log_scale=(False, True))
    axes[1].set_title("(b) Rozkład interakcji użytkowników")
    axes[1].set_xlabel("Liczba ocen wystawionych przez usera")
    axes[1].set_ylabel("Liczba użytkowników (log)")
    axes[1].grid(True, linestyle=':', alpha=0.6)
    
    # (e) Rozkład ocen od 1 do 5
    sns.countplot(data=df, x='rating', ax=axes[2], palette="viridis")
    axes[2].set_title("(e) Rozkład wystawianych ocen")
    axes[2].set_xlabel("Ocena (Gwiazdki)")
    axes[2].set_ylabel("Liczba")
    axes[2].grid(True, axis='y', linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

In [ ]:
analyze_dataset(df_full, "Pełny")
analyze_dataset(df_5core, "5-Core")

### Train-Test Split Strategies

Wybór strategi jest istotny bo musimy unikąc data-leakage

**1. Randomly select 20% of all interactions.**
* **Zalety:** Proste
* **Wady:** Możliwy wyciek danych, w zbiorze trenigowym mogą znaleźć się zakupy np z przyszłości względem danych testowych, tzn użytnownik ocenił kiedyś jakiś produkt i ta ocena mogła mieć wpływ na jego pryszłe zakupy.
* **Bias** W zbiorze tesotwym moze być domiacja uzywnowników, którzy kupuja dużo
* **Suitability:** Nie wydaje sie to być dobre rozwiązanie


**2. Randomly select 20% of users and put all their interactions in the test set.**
* **Zalety:** Brak wyciecku danych dla użytnokwinka, sprawdzay generalizajce modelu.
* **Wady:** Model nie nauczyc sie niczego o tych 20% użytoniwków, a w faktoryzjacji macierzy to istotne
* **Suitability:** Do rankingu wydaje sie byc nieprzydatne, ale gdybysmy nagle checieli cos powiedziec o duzej nowej grupie klinetów to moze miec sens


**3. Select 20% of interactions for each user at random.**
* **Zalety:** Każdy użytkownik jest uwzględniony zarówno w zbiorze treningowym, jak i testowym.
* **Wady:** Data leakage bo mieszamy przeszłość z przyszłością
* **Suitability:** No tez się nie nadaje

**4. Select the most recent 20% of interactions for each user.**
* **Zalety:** Brak wycieku danych z przyszłości do przeszłości dla konkretnego użytkownika i każdy użytnowik jest naturlanie reprezentowany, ALE MAMY LEKAGE bo uzytkownicy w różnych terminach korzystają
* **Wady:** Znowu dominacjia najbardziej aktywnych użytkowników
* **Suitability:** Dobra, ale nierównomiernie ocenia użytkowników.

**5. Select a fixed number of the most recent interactions for each user to obtain around 20% of the data.**
* **Zalety:** Każdy użytkonwik ma tą samą wage podczas tesotwania, brak leakage
* **Wady:** Wymaga uprzedniego odfiltrowania "rzadkich" użytkowników, bo jak ktoś ma za mało ocen to trafi jedynie do testowego
* **Suitability:** Wydaje sie być sensowne

**6. Select the most recent 20% of all interactions.**
* **Zalety:** Naturalne, bo symuluje po porstu testowanie modelu w setupie takim że nagle przychodzą użytkownicy i trzeba dla nich przetestować.
* **Wady:** Znowu byc może użytkoniwcy w testowym nie pojawią się w train.
* **Suitability:** Też wydaje się mieć sens

In [ ]:
print(f"Rozmiar zbioru 5-core przed podziałem: {len(df_5core)}")

df_5core_sorted = df_5core.sort_values(by=['user_id', 'timestamp'])

# Wyciągamy 2 najnowsze interakcje dla każdego użytkownika do zbioru testowego, reszta do train
test_df = df_5core_sorted.groupby('user_id').tail(2)
train_df = df_5core_sorted.drop(test_df.index)

print(f"Rozmiar zbioru treningowego (Train): {len(train_df)}")
print(f"Rozmiar zbioru testowego (Test): {len(test_df)}")

# sanity check
# rozmiar testu powinien być równy: liczba unikalnych userów * 2
unique_users = df_5core['user_id'].nunique()
if len(test_df) != (unique_users * 2):
    print("ZŁY SIZE")

### Boolean Feedback

Zamiana ocen na "boolean feedback" to przejście z systemu 1-5 gwiazdek na np. kupił / nie kupił, lubi / nie lubi. Prostrzych interakcji moze być prościej się nauczyć.

Możemy to zrobić na dwa logiczne sposoby:

1. **Interaction-based** Sama recenzja to już oznaka zaintresowania, tak samo sam zakup.

2. **Threshold-based** Ustwiamy, że np od oceny 4 polecamy a poniżej nie.

In [ ]:
# Oceny >= 4 stają się jedynkami (1), a reszta zerami (0)
train_df['boolean_feedback'] = (train_df['rating'] >= 4.0).astype(int)

In [ ]:
import numpy as np
from tqdm import tqdm

user_ids = df_5core['user_id'].unique()
item_ids = df_5core['asin'].unique()

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {i: idx for idx, i in enumerate(item_ids)}

train_df['user_idx'] = train_df['user_id'].map(user2idx)
train_df['item_idx'] = train_df['asin'].map(item2idx)

test_df['user_idx'] = test_df['user_id'].map(user2idx)
test_df['item_idx'] = test_df['asin'].map(item2idx)

print(f"Liczba unikalnych użytkowników: {len(user2idx)}")
print(f"Liczba unikalnych przedmiotów: {len(item2idx)}")

In [ ]:
print("\nTrenowanie modelu Popularity ")
# Model oparty na popularności to po prostu zliczenie wystąpień przedmiotów w zbiorze TRENINGOWYM
item_popularity = train_df['item_idx'].value_counts()
top_20_items = item_popularity.head(20).index.tolist()

print(f"Top 5 najpopularniejszych itemów (ID): {top_20_items[:5]}")

# Grupujemy zbiór testowy, żeby dla każdego usera mieć listę przedmiotów, które faktycznie kupił (GT)
test_user_items = test_df.groupby('user_idx')['item_idx'].apply(list).to_dict()

In [ ]:
def evaluate_model(recommendations_dict, test_ground_truth, k=20):
    """
    recommendations_dict:  {user_idx: [lista top-k przewidywanych item_idx]}
    test_ground_truth:  {user_idx: [lista prawdziwych item_idx z testu]}
    """
    hit_rate = []
    recall = []
    ndcg = []
    mrr = []
    
    for user_idx, true_items in test_ground_truth.items():
        if user_idx not in recommendations_dict:
            continue
            
        pred_items = recommendations_dict[user_idx][:k]
        
        # Hit-Rate@20 (Czy chociaż jeden kupiony przedmiot jest w top-20?)
        hits = len(set(pred_items) & set(true_items))
        hit_rate.append(1 if hits > 0 else 0)
        
        # Recall@20 (Ile z kupionych przedmiotó odgadliśmy?)
        recall.append(hits / len(true_items))
        
        # MRR@20 (Mean Reciprocal Rank - na którym miejscu był pierwszy trafiony?)
        user_mrr = 0
        for rank, item in enumerate(pred_items):
            if item in true_items:
                user_mrr = 1 / (rank + 1)
                break # bierzemy tylko pierwszy trafiony
        mrr.append(user_mrr)
        
        # NDCG@20 (Znormalizowana zdyskontowana funkcja zysku - zależy od pozycji)
        dcg = 0
        idcg = sum([1 / np.log2(i + 2) for i in range(min(len(true_items), k))])
        for rank, item in enumerate(pred_items):
            if item in true_items:
                dcg += 1 / np.log2(rank + 2)
        ndcg.append(dcg / idcg if idcg > 0 else 0)
        
    return {
        'HitRate@20': np.mean(hit_rate),
        'Recall@20': np.mean(recall),
        'MRR@20': np.mean(mrr),
        'NDCG@20': np.mean(ndcg)
    }

# Tworzymy rekomendacje z modelu Popularity dla każdego usera w teście
pop_recommendations = {user: top_20_items for user in test_user_items.keys()}

print("\nWyniki modelu Popularity (Baseline)")
pop_metrics = evaluate_model(pop_recommendations, test_user_items, k=20)
for metric, value in pop_metrics.items():
    print(f"{metric}: {value:.4f}")

Trochę słabe wyniki, średnio jeden na 100 ma na liście coś co realnie kupił.

# Task2

In [2]:
import os
import urllib.request
import zipfile
import pandas as pd
import numpy as np
import math
import time

if not os.path.exists('ml-100k'):
    print("Pobieranie danych...")
    urllib.request.urlretrieve('http://files.grouplens.org/datasets/movielens/ml-100k.zip', 'ml-100k.zip')
    with zipfile.ZipFile('ml-100k.zip', 'r') as zip_ref:
        zip_ref.extractall()

cols = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv('ml-100k/u.data', sep='\t', names=cols)

# Oceny >= 4 traktujemy jako "lubi to", resztę wyrzucamy
df = df[df['rating'] >= 4].copy()
df = df.sort_values(by=['user_id', 'timestamp'])

# Przekodowanie ID na ciągłe indeksy od 0 (wymagane przez PyTorch)
df['user_id'] = df['user_id'].astype('category').cat.codes
df['item_id'] = df['item_id'].astype('category').cat.codes

num_users = df['user_id'].nunique()
num_items = df['item_id'].nunique()
print(f"Liczba użytkowników: {num_users}")
print(f"Liczba przedmiotów: {num_items}")
print(f"Liczba pozytywnych interakcji: {len(df)}")

df['interaction_num'] = df.groupby('user_id').cumcount()
df['total_interactions'] = df.groupby('user_id')['item_id'].transform('count')

# wszystko od 80% w górę idzie do testu
is_test_mask = df['interaction_num'] >= (df['total_interactions'] * 0.8)

train_df = df[~is_test_mask].copy()
test_df = df[is_test_mask].copy()

train_df = train_df.drop(columns=['interaction_num', 'total_interactions'])
test_df = test_df.drop(columns=['interaction_num', 'total_interactions'])

print(f"Trening: {len(train_df)} interakcji")
print(f"Test: {len(test_df)} interakcji")

# user_id -> set(przedmioty z testu)
test_user_items = test_df.groupby('user_id')['item_id'].apply(set).to_dict()

Liczba użytkowników: 942
Liczba przedmiotów: 1447
Liczba pozytywnych interakcji: 55375
Trening: 44679 interakcji
Test: 10696 interakcji


In [3]:
def get_metrics(recommended_list, true_set, k=20):
    top_k = recommended_list[:k]
    
    # Hit Rate 
    hits = [1 if item in true_set else 0 for item in top_k]
    hit_rate = 1 if sum(hits) > 0 else 0
    
    # Recall
    recall = sum(hits) / len(true_set) if len(true_set) > 0 else 0
    
    # MRR: 1 / pozycja pierwszego trafienia
    mrr = 0
    for i, h in enumerate(hits):
        if h > 0:
            mrr = 1 / (i + 1)
            break
            
    # NDCG 
    dcg = 0
    for i, h in enumerate(hits):
        if h > 0:
            dcg += 1 / math.log2(i + 2) # i+2 bo ranga zaczyna się od 1
            
    # Idealne 
    idcg = sum([1 / math.log2(i + 2) for i in range(min(len(true_set), k))])
    ndcg = dcg / idcg if idcg > 0 else 0
    
    return hit_rate, recall, mrr, ndcg

In [4]:
# Popularity Baseline
popularity_counts = train_df['item_id'].value_counts()
# Sortujemy przedmioty od najpopularniejszego
global_popular_items = popularity_counts.index.tolist()
# Ewaluacja Baseline'u na zbiorze testowym
baseline_results = {'hr': [], 'recall': [], 'mrr': [], 'ndcg': []}

# Dla każdego użytkownika w teście sprawdzamy rekomendacje popularnych filmów
for user_id, true_items in test_user_items.items():
    # Pobieramy przedmioty, które użytkownik widział w treningu (żeby ich nie polecać)
    seen_in_train = set(train_df[train_df['user_id'] == user_id]['item_id'])
    
    # Rekomendujemy najpopularniejsze, których jeszcze nie widział
    recs = [item for item in global_popular_items if item not in seen_in_train]
    
    # Liczymy metryki @20
    hr, rec, mrr, ndcg = get_metrics(recs, true_items, k=20)
    
    baseline_results['hr'].append(hr)
    baseline_results['recall'].append(rec)
    baseline_results['mrr'].append(mrr)
    baseline_results['ndcg'].append(ndcg)

print("WYNIKI MODELU BAZOWEGO (MOST POPULAR)")
print(f"Hit-Rate@20: {np.mean(baseline_results['hr']):.4f}")
print(f"Recall@20:   {np.mean(baseline_results['recall']):.4f}")
print(f"MRR@20:      {np.mean(baseline_results['mrr']):.4f}")
print(f"NDCG@20:     {np.mean(baseline_results['ndcg']):.4f}")

WYNIKI MODELU BAZOWEGO (MOST POPULAR)
Hit-Rate@20: 0.5181
Recall@20:   0.1159
MRR@20:      0.1747
NDCG@20:     0.0942


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
import random

class BPRDataset(Dataset):
    def __init__(self, df, num_items):
        self.users = df['user_id'].values
        self.pos_items = df['item_id'].values
        self.num_items = num_items
        
        # Słownik: user -> set(przedmioty widziane w treningu)
        # Bardzo ważne: zapobiega wylosowaniu jako "negatywu" filmu, który użytkownik obejrzał
        self.user_history = df.groupby('user_id')['item_id'].apply(set).to_dict()

    def __len__(self):
        # Ilość danych treningowych (każda pozytywna interakcja to jeden wiersz)
        return len(self.users)

    def __getitem__(self, idx):
        u = self.users[idx]
        pos_i = self.pos_items[idx]
        
        # UNIFORM NEGATIVE SAMPLING
        # Losujemy dowolne ID przedmiotu
        neg_i = random.randint(0, self.num_items - 1)
        # Jeśli wylosowaliśmy coś, co użytkownik już zna, losujemy ponownie
        while neg_i in self.user_history[u]:
            neg_i = random.randint(0, self.num_items - 1)
            
        # Zwracamy trójkę gotową do obliczeń w BPR (użytkownik, pozytyw, negatyw)
        return torch.tensor(u, dtype=torch.long), \
               torch.tensor(pos_i, dtype=torch.long), \
               torch.tensor(neg_i, dtype=torch.long)

BATCH_SIZE = 1024 

train_dataset = BPRDataset(train_df, num_items)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Utworzono DataLoader.")
print(f"Rozmiar mini-batcha: {BATCH_SIZE}")
print(f"Liczba batchy w jednej epoce: {len(train_loader)}")

Utworzono DataLoader.
Rozmiar mini-batcha: 1024
Liczba batchy w jednej epoce: 44


Matrix factorization:

$$\hat{r}_{ui} = b_u + b_i + \mathbf{p}_u^T \mathbf{q}_i$$

In [9]:
import torch.nn as nn
import torch.nn.functional as F


import torch
import torch.nn as nn
import torch.nn.functional as F

class MatrixFactorizationBPR(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim):
        super(MatrixFactorizationBPR, self).__init__()
        
        # Główne embeddingi (P i Q)
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        
        # Biasy (b_u i b_i) - wymiar 1
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight) 
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, u, pos_i, neg_i):
        p_u = self.user_emb(u)
        q_pos = self.item_emb(pos_i)
        q_neg = self.item_emb(neg_i)
        
        b_u = self.user_bias(u).squeeze()
        b_pos = self.item_bias(pos_i).squeeze()
        b_neg = self.item_bias(neg_i).squeeze()
        
        pos_scores = (p_u * q_pos).sum(dim=1) + b_u + b_pos
        neg_scores = (p_u * q_neg).sum(dim=1) + b_u + b_neg
        
        return pos_scores, neg_scores, p_u, q_pos, q_neg, b_u, b_pos, b_neg

    def get_all_scores(self, u):
        p_u = self.user_emb(u) 
        b_u = self.user_bias(u)
        
        scores = torch.matmul(p_u, self.item_emb.weight.T) + b_u + self.item_bias.weight.T
        return scores

def bpr_loss_fn(pos_scores, neg_scores, p_u, q_pos, q_neg, b_u, b_pos, b_neg, lambda_reg=1e-4):
    bpr_loss = -torch.mean(F.logsigmoid(pos_scores - neg_scores))
    
    batch_size = p_u.size(0)
    
    l2_reg = (p_u.norm(2).pow(2) + 
              q_pos.norm(2).pow(2) + 
              q_neg.norm(2).pow(2) + 
              b_u.norm(2).pow(2) + 
              b_pos.norm(2).pow(2) + 
              b_neg.norm(2).pow(2))
    
    return bpr_loss + (lambda_reg * l2_reg / batch_size)

In [7]:
DEVICE = torch.device("mps" if torch.mps.is_available() else "cpu")
print(f"Używane urządzenie: {DEVICE}")

EPOCHS = 10
LEARNING_RATE = 0.01
LAMBDA_REG = 1e-4
EMBEDDING_DIMS = [16, 32, 64, 128]

Używane urządzenie: mps


In [10]:
final_results = {}

for dim in EMBEDDING_DIMS:
    print(f"\n{'='*40}")
    print(f"ROZPOCZĘCIE TRENINGU DLA WYMIARU: {dim}")
    print(f"{'='*40}")
    
    model = MatrixFactorizationBPR(num_users, num_items, dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        
        model.train()
        total_loss = 0.0
        
        for u, pos_i, neg_i in train_loader:
            u, pos_i, neg_i = u.to(DEVICE), pos_i.to(DEVICE), neg_i.to(DEVICE)
            
            optimizer.zero_grad()
            
            pos_scores, neg_scores, p_u, q_pos, q_neg, b_u, b_pos, b_neg = model(u, pos_i, neg_i)
            
            loss = bpr_loss_fn(pos_scores, neg_scores, p_u, q_pos, q_neg, b_u, b_pos, b_neg, lambda_reg=LAMBDA_REG)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        avg_loss = total_loss / len(train_loader)
        
        model.eval()
        epoch_metrics = {'hr': [], 'recall': [], 'mrr': [], 'ndcg': []}
        
        with torch.no_grad():
            for user_id, true_items in test_user_items.items():
                u_tensor = torch.tensor([user_id], dtype=torch.long).to(DEVICE)
                
                scores = model.get_all_scores(u_tensor).squeeze()
                
                seen_items = list(train_dataset.user_history.get(user_id, set()))
                if seen_items:
                    scores[seen_items] = -float('inf') 
                
                top_20_idx = torch.topk(scores, 20).indices.cpu().tolist()
                
                hr, rec, mrr, ndcg = get_metrics(top_20_idx, true_items, k=20)
                epoch_metrics['hr'].append(hr)
                epoch_metrics['recall'].append(rec)
                epoch_metrics['mrr'].append(mrr)
                epoch_metrics['ndcg'].append(ndcg)
                
        mean_hr = np.mean(epoch_metrics['hr'])
        mean_rec = np.mean(epoch_metrics['recall'])
        mean_mrr = np.mean(epoch_metrics['mrr'])
        mean_ndcg = np.mean(epoch_metrics['ndcg'])
        
        epoch_time = time.time() - start_time
        print(f"Epoka {epoch+1:02d}/{EPOCHS} | Czas: {epoch_time:.1f}s | Loss: {avg_loss:.4f} | HR@20: {mean_hr:.4f} | Rec@20: {mean_rec:.4f} | NDCG@20: {mean_ndcg:.4f} | MRR@20: {mean_mrr:.4f}")
        
    final_results[dim] = {
        'HR@20': mean_hr,
        'Recall@20': mean_rec,
        'NDCG@20': mean_ndcg,
        'MRR@20': mean_mrr
    }


ROZPOCZĘCIE TRENINGU DLA WYMIARU: 16
Epoka 01/10 | Czas: 2.6s | Loss: 0.5943 | HR@20: 0.5224 | Rec@20: 0.1164 | NDCG@20: 0.0971 | MRR@20: 0.1810
Epoka 02/10 | Czas: 1.7s | Loss: 0.3290 | HR@20: 0.5906 | Rec@20: 0.1438 | NDCG@20: 0.1149 | MRR@20: 0.2117
Epoka 03/10 | Czas: 2.4s | Loss: 0.2561 | HR@20: 0.6045 | Rec@20: 0.1551 | NDCG@20: 0.1213 | MRR@20: 0.2175
Epoka 04/10 | Czas: 2.1s | Loss: 0.2265 | HR@20: 0.6162 | Rec@20: 0.1657 | NDCG@20: 0.1238 | MRR@20: 0.2097
Epoka 05/10 | Czas: 1.7s | Loss: 0.2062 | HR@20: 0.6269 | Rec@20: 0.1693 | NDCG@20: 0.1260 | MRR@20: 0.2060
Epoka 06/10 | Czas: 1.5s | Loss: 0.1887 | HR@20: 0.6343 | Rec@20: 0.1754 | NDCG@20: 0.1306 | MRR@20: 0.2119
Epoka 07/10 | Czas: 2.5s | Loss: 0.1784 | HR@20: 0.6439 | Rec@20: 0.1798 | NDCG@20: 0.1330 | MRR@20: 0.2138
Epoka 08/10 | Czas: 2.1s | Loss: 0.1717 | HR@20: 0.6439 | Rec@20: 0.1810 | NDCG@20: 0.1346 | MRR@20: 0.2169
Epoka 09/10 | Czas: 1.6s | Loss: 0.1626 | HR@20: 0.6674 | Rec@20: 0.1907 | NDCG@20: 0.1403 | MRR@2

In [11]:
print("\n" + "="*50)
print("OSTATECZNE PORÓWNANIE ROZMIARÓW EMBEDDINGÓW")
print("="*50)
print(f"{'Dim':<5} | {'HR@20':<8} | {'Recall@20':<10} | {'NDCG@20':<8} | {'MRR@20':<8}")
print("-" * 50)
for dim, metrics in final_results.items():
    print(f"{dim:<5} | {metrics['HR@20']:.4f}   | {metrics['Recall@20']:.4f}     | {metrics['NDCG@20']:.4f}   | {metrics['MRR@20']:.4f}")


OSTATECZNE PORÓWNANIE ROZMIARÓW EMBEDDINGÓW
Dim   | HR@20    | Recall@20  | NDCG@20  | MRR@20  
--------------------------------------------------
16    | 0.6674   | 0.1901     | 0.1417   | 0.2224
32    | 0.6674   | 0.1836     | 0.1376   | 0.2259
64    | 0.6962   | 0.1886     | 0.1420   | 0.2330
128   | 0.6823   | 0.1861     | 0.1406   | 0.2369


In [12]:
class MatrixFactorizationMSE(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim):
        super(MatrixFactorizationMSE, self).__init__()
        
        # Embeddingi wektorów cech (P i Q)
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, u, i):
        p_u = self.user_emb(u)
        q_i = self.item_emb(i)
        
        b_u = self.user_bias(u).squeeze()
        b_i = self.item_bias(i).squeeze()
        
        dot_product = (p_u * q_i).sum(dim=1)
        
        prediction = torch.sigmoid(dot_product + b_u + b_i)
        
        return prediction, p_u, q_i, b_u, b_i

def mse_loss_fn(preds, targets, p_u, q_i, b_u, b_i, lambda_reg=1e-4):
    mse = F.mse_loss(preds, targets, reduction='sum')
    
    reg = lambda_reg * (
        p_u.norm(2).pow(2) + 
        q_i.norm(2).pow(2) + 
        b_u.norm(2).pow(2) + 
        b_i.norm(2).pow(2)
    )
    
    return (mse + reg) / preds.size(0)

train_df_mse = train_df.copy()
train_df_mse['rating_norm'] = (train_df_mse['rating'] - 1) / (5 - 1)

class MSEDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_id'].values, dtype=torch.long)
        self.items = torch.tensor(df['item_id'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating_norm'].values, dtype=torch.float)
    def __len__(self): return len(self.users)
    def __getitem__(self, idx): return self.users[idx], self.items[idx], self.ratings[idx]

mse_loader = DataLoader(MSEDataset(train_df_mse), batch_size=BATCH_SIZE, shuffle=True)

In [13]:
class MatrixFactorizationAU(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim):
        super(MatrixFactorizationAU, self).__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, u, i):
        p_u = F.normalize(self.user_emb(u), p=2, dim=1)
        q_i = F.normalize(self.item_emb(i), p=2, dim=1)
        
        b_u = self.user_bias(u).squeeze()
        b_i = self.item_bias(i).squeeze()
        
        return p_u, q_i, b_u, b_i

def au_loss_fn(p_u, q_i, b_u, b_i, lambda_weight=1.0, lambda_reg=1e-4):
    dot_product = (p_u * q_i).sum(dim=1)
    score = dot_product + b_u + b_i
    
    alignment = -score.mean()
    
    def uniformity(x):
        dist_sq = torch.cdist(x, x, p=2).pow(2)
        return torch.log(torch.exp(-2.0 * dist_sq).mean() + 1e-8)
    
    reg_loss = lambda_reg * (b_u.norm(2).pow(2) + b_i.norm(2).pow(2))
    
    u_loss = uniformity(p_u) + uniformity(q_i)
    
    return alignment + (lambda_weight * u_loss) + reg_loss

class AUDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_id'].values, dtype=torch.long)
        self.items = torch.tensor(df['item_id'].values, dtype=torch.long)
    def __len__(self): return len(self.users)
    def __getitem__(self, idx): return self.users[idx], self.items[idx]

au_loader = DataLoader(AUDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)

In [14]:
LOSS_TYPES = ['BPR', 'MSE', 'AU']
EMBEDDING_DIMS = [16, 32, 64, 128]
EPOCHS = 20  
LR = 0.01

In [19]:
all_final_results = []

for loss_name in LOSS_TYPES:
    print(f"\n\nTESTOWANIE FUNKCJI STRATY: {loss_name}")
    
    if loss_name == 'BPR':
        current_loader = train_loader 
    elif loss_name == 'MSE':
        current_loader = mse_loader   
    else: 
        current_loader = au_loader    

    for dim in EMBEDDING_DIMS:
        print(f"Trenowanie {loss_name} | Dim: {dim}")
        
        if loss_name == 'BPR':
            model = MatrixFactorizationBPR(num_users, num_items, dim).to(DEVICE)
        elif loss_name == 'MSE':
            model = MatrixFactorizationMSE(num_users, num_items, dim).to(DEVICE)
        else:
            model = MatrixFactorizationAU(num_users, num_items, dim).to(DEVICE)
            
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        
        start_train = time.time()
        
        for epoch in range(EPOCHS):
            model.train()

            running_loss = {}

            for batch in current_loader:
                optimizer.zero_grad()
                
                if loss_name == 'BPR':
                    u, i, j = [x.to(DEVICE) for x in batch]
                    pos_s, neg_s, p_u, q_i, q_j, b_u, b_pos, b_n = model(u, i, j) 
                    loss = bpr_loss_fn(pos_s, neg_s, p_u, q_i, q_j, b_u, b_pos, b_n)
                elif loss_name == 'MSE':
                    u, i, r = [x.to(DEVICE) for x in batch]
                    preds, p_u, q_i, b_u, b_i = model(u, i)  # TUTAJ
                    loss = mse_loss_fn(preds, r, p_u, q_i, b_u, b_i)
                else: # AU
                    u, i = [x.to(DEVICE) for x in batch]
                    p_u, q_i, b_u, b_i = model(u, i)
                    loss = au_loss_fn(p_u, q_i, b_u, b_i)
                
                loss.backward()
                optimizer.step()

                running_loss[loss_name] = loss.item()
                
            if epoch % 5 == 0:
                print(f"Epochs: {epoch + 1}: {running_loss}")
        
        train_time = time.time() - start_train
        
        # EWAL
        model.eval()
        metrics_list = {'hr': [], 'rec': [], 'mrr': [], 'ndcg': []}
        
        with torch.no_grad():
            for user_id, true_items in test_user_items.items():
                u_tensor = torch.tensor([user_id], dtype=torch.long).to(DEVICE)
                
                if loss_name == 'MSE':
                    p_u = model.user_emb(u_tensor)
                    scores = torch.sigmoid(torch.matmul(p_u, model.item_emb.weight.T)).squeeze()
                elif loss_name == 'AU':
                    p_u = F.normalize(model.user_emb(u_tensor), p=2, dim=1)
                    q_items = F.normalize(model.item_emb.weight, p=2, dim=1)
                    scores = torch.matmul(p_u, q_items.T).squeeze()
                else: # BPR
                    scores = model.get_all_scores(u_tensor).squeeze()
                
                # nie polecamy tego co było w treningu
                seen = list(train_dataset.user_history.get(user_id, set()))
                scores[seen] = -float('inf')
                
                top_20 = torch.topk(scores, 20).indices.cpu().tolist()
                hr, rec, mrr, ndcg = get_metrics(top_20, true_items, k=20)
                
                metrics_list['hr'].append(hr)
                metrics_list['rec'].append(rec)
                metrics_list['mrr'].append(mrr)
                metrics_list['ndcg'].append(ndcg)
        
        res = {
            'Loss': loss_name,
            'Dim': dim,
            'HR@20': np.mean(metrics_list['hr']),
            'Recall@20': np.mean(metrics_list['rec']),
            'NDCG@20': np.mean(metrics_list['ndcg']),
            'MRR@20': np.mean(metrics_list['mrr']),
            'TrainTime': train_time
        }
        all_final_results.append(res)
        print(f"Zakończono: HR@20 = {res['HR@20']:.4f}, Czas: {train_time:.1f}s")



TESTOWANIE FUNKCJI STRATY: BPR
Trenowanie BPR | Dim: 16
Epochs: 1: {'BPR': 0.40955203771591187}
Epochs: 6: {'BPR': 0.17250971496105194}
Epochs: 11: {'BPR': 0.16153880953788757}
Epochs: 16: {'BPR': 0.15585553646087646}
Zakończono: HR@20 = 0.6770, Czas: 12.8s
Trenowanie BPR | Dim: 32
Epochs: 1: {'BPR': 0.37839475274086}
Epochs: 6: {'BPR': 0.15525676310062408}
Epochs: 11: {'BPR': 0.12230886518955231}
Epochs: 16: {'BPR': 0.07267747819423676}
Zakończono: HR@20 = 0.6908, Czas: 12.2s
Trenowanie BPR | Dim: 64
Epochs: 1: {'BPR': 0.3302474319934845}
Epochs: 6: {'BPR': 0.1333450824022293}
Epochs: 11: {'BPR': 0.09088445454835892}
Epochs: 16: {'BPR': 0.06869020313024521}
Zakończono: HR@20 = 0.6876, Czas: 12.6s
Trenowanie BPR | Dim: 128
Epochs: 1: {'BPR': 0.3082134425640106}
Epochs: 6: {'BPR': 0.10105150938034058}
Epochs: 11: {'BPR': 0.04871336743235588}
Epochs: 16: {'BPR': 0.058204296976327896}
Zakończono: HR@20 = 0.6962, Czas: 13.6s


TESTOWANIE FUNKCJI STRATY: MSE
Trenowanie MSE | Dim: 16
Epoch

In [20]:
summary_df = pd.DataFrame(all_final_results)
print("\n\n" + "="*60)
print("PEŁNE PORÓWNANIE MODELI I PARAMETRÓW")
print("="*60)
print(summary_df.to_string(index=False))



PEŁNE PORÓWNANIE MODELI I PARAMETRÓW
Loss  Dim    HR@20  Recall@20  NDCG@20   MRR@20  TrainTime
 BPR   16 0.676972   0.190063 0.141146 0.221097  12.782293
 BPR   32 0.690832   0.185660 0.139089 0.224351  12.163701
 BPR   64 0.687633   0.185356 0.138772 0.233663  12.557365
 BPR  128 0.696162   0.187966 0.135283 0.218364  13.576329
 MSE   16 0.246269   0.032325 0.031154 0.078798   7.645795
 MSE   32 0.279318   0.036862 0.031924 0.074335   7.381905
 MSE   64 0.320896   0.042275 0.037481 0.092995   7.437992
 MSE  128 0.298507   0.041268 0.033748 0.077226   7.978103
  AU   16 0.344350   0.059781 0.042201 0.082864  13.872561
  AU   32 0.284648   0.046271 0.032997 0.066112  13.693005
  AU   64 0.229211   0.049755 0.029464 0.044328  14.460797
  AU  128 0.224947   0.049287 0.028561 0.041517  13.409537


Najlepszy: Tylko BPR (HR@20: 0.70) pobił model popularności (0.51).

Najlepszy wymiar: 32 jest najlepszy dla BPR

Porażka MSE i AU: Oba modele wypadły słabo (HR poniżej 0.40), dając gorsze wyniki niż polecanie hitów „na ślepo”.